# HeatUp 01 - structure preparation and MD

Prepares the inputs of the stability gates. Every step is cached (`manifest.json` = config fingerprint +
input hashes), so re-running a cell only recomputes what changed. Each step runs in its own process.

| step | output (per `<material>/<label>/`) | used by | what it does |
|---|---|---|---|
| `relax` | `relaxation/CONTCAR`, `energy.json` | everything | BFGS geometry optimisation |
| `elastic` | `elastic/elastic_tensor.json` (relaxed-ion) | Gate 1 | 6×6 Voigt stiffness tensor |
| `phonons` | `phonons/ha.json` (harmonic modes) | Gate 2/3 | Harmonic DOS (always) + QHA if configured |
| `qha` (optional) | `phonons/qha.json` | Gate 3 (`hull.phonon_method="QHA"`) | VACF → anharmonic phonon DOS and F(T) |
| `md` | `md/<T>K/output.traj` (positions, momenta, energy, stress) | Gate 0/2, VDOS | NPT or NVT trajectory |

## Where the data lives
```
<database_root>/                    targets: read + write
└── <material>/<label>/             e.g. BaTiO3/P4mm
    ├── POSCAR                      the input structure
    ├── relaxation/   CONTCAR, energy.json, run.traj          step "relax"
    ├── elastic/      elastic.json                            step "elastic"   -> Gate 1
    ├── phonons/      ha.json, [qha.json], phonon.png         steps "phonons", "qha"
    ├── md/<T>K/      output.traj, ...                        step "md"        -> Gates 0, 2
    └── stability/    report_<stage>.json, result_<stage>.json, report_<stage>.png, zero_K_hull.json
<candidates_root>/                  user inputs: read-only, never written
<competing_root>/                   competing phases (MP downloads, generated, mirrored): same <material>/<label>/ layout,
                                    each with its own relaxation/ and phonons/ (they feed Gate 3)
```
Every step directory carries a `manifest.json` (config fingerprint + input hashes): a step is recomputed when its
inputs or relevant options changed, or when it is listed in `cfg.run.force`.

In [ ]:
import os, warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import heatup
from heatup import cfg, run_step, PhaseDir, override
from heatup.runtime import verbose
from heatup.report import print_layout

warnings.filterwarnings("ignore")
verbose()
print("HeatUp", heatup.__version__)
from heatup.builders import locate_phase, write_phase, fcc
from heatup import database_status

## Configuration

In [ ]:
# --- calculator -----------------------------------------------------------
cfg.calc.backend = os.environ.get("HEATUP_BACKEND", "mace-mp")
cfg.calc.model = os.environ.get("MACE_MODEL_PATH", "medium-mpa-0")  # mace_mp name or path to a .model
cfg.calc.model_overrides = {}                                       # e.g. {"BaTiO3": "/path/BTO_MACE.model"}
DEVICE = None                                                       # None -> cuda if usable, else cpu

cfg.phases.database_root = "database"
cfg.phases.candidates_root = "input/candidates"  # read-only

cfg.md.ensemble = "NPT"
cfg.md.n_steps = 30_000  # 30 ps at 1 fs; the VDOS wants >= gates.min_production_ps after equilibration
cfg.md.equil_ps = 5.0
cfg.md.min_cell_A = 12.0

RUN = ["relax", "elastic", "phonons", "md"]  # add "qha" if hull.phonon_method = "QHA"
cfg.run.force = []                           # steps to recompute whatever the cache says: ["md"], ["relax", "elastic"], "all"
RUN_LIST = [                                 # (material folder, label, [MD temperatures in K])
    ("Cu", "Fm-3m", [300]),
]

## Locate the inputs
A POSCAR in `database/<material>/<label>/` is used as is; one found only under `input/candidates/` is copied into the database.

In [ ]:
phases = []
for mat, label, temps in RUN_LIST:
    d = locate_phase(mat, label)
    if d is None and cfg.calc.backend == "emt" and mat == "Cu":          # smoke-test fallback
        d = write_phase(cfg.phases.database_root, mat, label, fcc("Cu", 3.61))
    if d is None:
        print(f"[missing] {mat}/{label}: put a POSCAR in {cfg.phases.database_root}/{mat}/{label}/")
        continue
    phases.append((d, temps))
print(len(phases), "phase(s) ready")

## Run
A failed step prints a message and the loop moves on to the next phase.

In [ ]:
for d, temps in phases:
    print("=" * 70, "\n", d.tag)
    for step in RUN:
        if step == "md":
            ok = all(run_step("md", d, device=DEVICE, temperature=T) for T in temps)
        else:
            ok = run_step(step, d, device=DEVICE)
        if not ok:
            print(f"  [{step}] failed - see the log above; later steps need it")
            break

## What exists now

In [ ]:
display(pd.DataFrame(database_status(cfg.phases.database_root)))
print_layout(cfg.phases.database_root)

### Forcing a rerun
`cfg.run.force` lists the steps to recompute (`"all"` for every step); `run_step(..., force=True)` does it for one call. Cached steps print `[cached]`.

In [ ]:
with override(**{"run.force": ["relax"]}):                  # recompute only the relaxation, once
    run_step("relax", phases[0][0], device=DEVICE)

## Provenance of the cached results

In [ ]:
from heatup.manifest import manifest_summary
print(manifest_summary(phases[0][0].join("relaxation")))